# Dynamic Knapsack Team Formation (Using Pre-Extracted Skills)

This notebook loads the **uploaded** datasets:

- `/mnt/data/large_researcher_skills.csv`
- `/mnt/data/large_proposal_skills.csv`

and runs a **dynamic / marginal-utility knapsack** team formation algorithm using **IDF-style skill rarity** weights.

> You can export results to CSVs for downstream analysis.


In [1]:

# === 1) SETUP ===
import pandas as pd
import numpy as np
import ast
import math
import random
from collections import Counter, defaultdict

RESEARCHER_SKILLS_PATH = "../data/input_data/Set_4/large_researcher_skills.csv"
PROPOSAL_SKILLS_PATH   = "../data/input_data/Set_4/large_proposal_skills.csv"


RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print("Loaded paths:")
print(" -", RESEARCHER_SKILLS_PATH)
print(" -", PROPOSAL_SKILLS_PATH)


Loaded paths:
 - ../data/input_data/Set_4/large_researcher_skills.csv
 - ../data/input_data/Set_4/large_proposal_skills.csv


In [2]:

# === 2) LOAD DATASETS ===
researchers_df = pd.read_csv(RESEARCHER_SKILLS_PATH)
proposals_df   = pd.read_csv(PROPOSAL_SKILLS_PATH)

print("Researchers:", researchers_df.shape)
print("Proposals:  ", proposals_df.shape)

display(researchers_df.head(3))
display(proposals_df.head(3))


Researchers: (2000, 3)
Proposals:   (500, 3)


,Unnamed: 0,researcher_name,skills
0,0,Carmen Meyer,"{'ceramics', 'artificial intelligence', 'neuro..."
1,1,Danielle Rodriguez PhD,"{'project management', 'graph theory', 'biolog..."
2,2,Tyler Hill DVM,"{'artificial intelligence', 'deep learning', '..."


,Unnamed: 0,nsf_proposal_links_v0,skills
0,0,https://www.nsf.gov/pubs/2026/nsf26000/nsf2600...,"{'python', 'optimization', 'deep learning', 's..."
1,1,https://www.nsf.gov/pubs/2026/nsf26001/nsf2600...,"{'data science', 'python', 'deep learning', 'n..."
2,2,https://www.nsf.gov/pubs/2026/nsf26002/nsf2600...,"{'structural health monitoring', 'machine lear..."


In [3]:

# === 3) PARSE SKILL SETS ===
# The `skills` column is a string like: "{'python', 'deep learning', ...}"
# We'll parse it into Python sets.

def parse_skill_set(x):
    if pd.isna(x):
        return set()
    if isinstance(x, (set, list, tuple)):
        return set(x)
    try:
        val = ast.literal_eval(x)
        return set(val) if isinstance(val, (set, list, tuple)) else set()
    except Exception:
        # Fallback: try splitting crudely (rarely needed)
        s = str(x).strip()
        s = s.strip("{}")
        parts = [p.strip().strip("'").strip('"') for p in s.split(",") if p.strip()]
        return set(parts)

researchers_df["skill_set"] = researchers_df["skills"].apply(parse_skill_set)
proposals_df["skill_set"]   = proposals_df["skills"].apply(parse_skill_set)

print("Parsed skill sets.")
print("Example researcher skill count:", len(researchers_df.loc[0, "skill_set"]))
print("Example proposal skill count:  ", len(proposals_df.loc[0, "skill_set"]))


Parsed skill sets.
Example researcher skill count: 4
Example proposal skill count:   4


In [4]:

# === 4) BUILD LOOKUPS ===
# Convert to the exact structures your solver expects:
# - m1_researcher_skills: dict[name] -> set(skills)
# - m1_proposal_skills:   dict[proposal_link] -> set(skills)

RESEARCHER_NAME_COL = "researcher_name"
PROPOSAL_LINK_COL   = "nsf_proposal_links_v0"

m1_researcher_skills = dict(zip(researchers_df[RESEARCHER_NAME_COL], researchers_df["skill_set"]))
m1_proposal_skills   = dict(zip(proposals_df[PROPOSAL_LINK_COL], proposals_df["skill_set"]))

all_researchers = list(m1_researcher_skills.keys())
all_proposals   = list(m1_proposal_skills.keys())

print("Researchers in map:", len(m1_researcher_skills))
print("Proposals in map:  ", len(m1_proposal_skills))
print("Sample researcher:", all_researchers[0])
print("Sample proposal:  ", all_proposals[0])


Researchers in map: 1968
Proposals in map:   500
Sample researcher: Carmen Meyer
Sample proposal:   https://www.nsf.gov/pubs/2026/nsf26000/nsf26000.htm


In [5]:

# === 5) SKILL RARITY WEIGHTS (IDF) ===
# Weight = log(TotalResearchers / (freq + 1)) + 1
# Rare skills -> higher weight.

skill_counts = Counter()
for r, skills in m1_researcher_skills.items():
    skill_counts.update(skills)

total_researchers = len(m1_researcher_skills)

skill_weights = {}
for s, count in skill_counts.items():
    skill_weights[s] = math.log(total_researchers / (count + 1)) + 1

DEFAULT_WEIGHT = 1.0

# Quick peek
common = skill_counts.most_common(10)
rare   = sorted(skill_counts.items(), key=lambda x: x[1])[:10]

print("Most common skills (count):")
for s, c in common:
    print(f"  {s:35s} {c:5d}  weight={skill_weights.get(s, DEFAULT_WEIGHT):.3f}")

print("\nRarest skills (count):")
for s, c in rare:
    print(f"  {s:35s} {c:5d}  weight={skill_weights.get(s, DEFAULT_WEIGHT):.3f}")


Most common skills (count):
  logistics                             228  weight=3.151
  applied mathematics                   225  weight=3.164
  biophysics                            223  weight=3.173
  software engineering                  222  weight=3.178
  business                              221  weight=3.182
  python                                220  weight=3.187
  geotechnical engineering              219  weight=3.191
  optimization                          218  weight=3.196
  robotics                              216  weight=3.205
  economics                             216  weight=3.205

Rarest skills (count):
  pedagogy                              176  weight=3.409
  structural health monitoring          179  weight=3.392
  polymers                              180  weight=3.386
  mathematics                           180  weight=3.386
  artificial intelligence               184  weight=3.364
  climate change                        186  weight=3.354
  material science  

In [6]:

# === 6) DYNAMIC KNAPSACK SOLVER ===
def solve_dynamic_knapsack(
    target_r,
    p_link,
    pseudo_skills_map,
    capacity_limit=8,
    gain_threshold=0.0,
    stochastic=False,
    cost_multiplier=0.05,
):
    """
    Principled Information-Theoretic Knapsack:
    - Adaptive Friction: cost based on the proposal's average skill weight.
    - Diminishing Returns: each extra coverage halves marginal value (0.5^n).
    """
    current_team = [target_r]
    req_skills = m1_proposal_skills.get(p_link, set())
    if not req_skills:
        return current_team, {"cost_constant": 0.0, "coverage_counts": {}, "req_skills": set()}

    # Local market cost
    avg_skill_weight = sum(skill_weights.get(s, DEFAULT_WEIGHT) for s in req_skills) / max(1, len(req_skills))
    cost_constant = avg_skill_weight * cost_multiplier

    coverage_counts = {skill: 0 for skill in req_skills}

    # seed coverage by target researcher
    target_skills = set(pseudo_skills_map.get(target_r, set())).intersection(req_skills)
    for skill in target_skills:
        coverage_counts[skill] += 1

    while len(current_team) < capacity_limit:
        candidates = []
        available_pool = [r for r in all_researchers if r not in current_team]

        for cand in available_pool:
            cand_skills = set(pseudo_skills_map.get(cand, set())).intersection(req_skills)
            total_cand_utility = 0.0

            for skill in cand_skills:
                n = coverage_counts[skill]
                weight = skill_weights.get(skill, DEFAULT_WEIGHT)
                total_cand_utility += weight * (math.pow(0.5, n))  # diminishing returns

            marginal_utility = total_cand_utility - cost_constant
            if marginal_utility > 0:
                candidates.append((cand, marginal_utility))

        if not candidates:
            break

        candidates.sort(key=lambda x: x[1], reverse=True)

        if not stochastic:
            best_cand, max_utility = candidates[0]
        else:
            top_pool = candidates[: min(3, len(candidates))]
            best_cand, max_utility = random.choice(top_pool)

        if max_utility > gain_threshold:
            current_team.append(best_cand)
            new_skills = set(pseudo_skills_map.get(best_cand, set())).intersection(req_skills)
            for s in new_skills:
                coverage_counts[s] += 1
        else:
            break

    debug = {"cost_constant": cost_constant, "coverage_counts": coverage_counts, "req_skills": req_skills}
    return current_team, debug


In [7]:
# ==========================================
# (ADD THIS) IMPORT M1 + GOODNESS (ULTRA METRIC)
# ==========================================
import os, sys

# If M1.py is in the same folder as your notebook/script, this is enough:
import M1

# If you get "ModuleNotFoundError: No module named 'M1'", uncomment and set the path:
# sys.path.append("/path/to/your/project/code")  # folder that contains M1.py
# import M1


# ==========================================
# === 7) PICK A TARGET RESEARCHER (DEFAULT: best single-person overlap) ===
# ==========================================
def pick_best_target_for_proposal(p_link):
    req = m1_proposal_skills.get(p_link, set())
    if not req:
        return random.choice(all_researchers)

    best_r = None
    best_score = -1.0

    for r in all_researchers:
        overlap = m1_researcher_skills[r].intersection(req)
        score = sum(skill_weights.get(s, DEFAULT_WEIGHT) for s in overlap)
        if score > best_score:
            best_score = score
            best_r = r

    return best_r if best_r is not None else random.choice(all_researchers)


# ==========================================
# DEMO ON ONE PROPOSAL (NOW PRINTS ULTRA METRIC GOODNESS)
# ==========================================
demo_p = all_proposals[0]
target = pick_best_target_for_proposal(demo_p)

team, debug = solve_dynamic_knapsack(
    target_r=target,
    p_link=demo_p,
    pseudo_skills_map=m1_researcher_skills,   # using uploaded skills as the pseudo skill map
    capacity_limit=8,
    stochastic=True
)

print("Proposal:", demo_p)
print("Target researcher:", target)
print("Team size:", len(team))
print("Team:", team)
print("Cost constant:", round(debug["cost_constant"], 4))

# Coverage report
req = debug["req_skills"]
covered = set().union(*[m1_researcher_skills[r].intersection(req) for r in team])
print("Req skills:", len(req))
print("Covered skills:", len(covered))
print("Coverage %:", round(100 * len(covered) / max(1, len(req)), 2))

# Ultra-metric goodness (from M1)
# Signature expected: apply_ultra_metric(req_skills_set, team_list, pseudo_skills_map)
goodness_ultra = M1.apply_ultra_metric(req, team, m1_researcher_skills)
print("Goodness (M1.apply_ultra_metric):", round(goodness_ultra, 4))


# ==========================================
# === 8) RUN TEAMING FOR ALL PROPOSALS (NOW INCLUDES GOODNESS)
# ==========================================
def run_teaming_for_proposals(
    proposal_links,
    capacity_limit=8,
    stochastic=True,
    gain_threshold=0.0,
    cost_multiplier=0.05
):
    results = []
    for idx, p_link in enumerate(proposal_links):
        target = pick_best_target_for_proposal(p_link)

        team, debug = solve_dynamic_knapsack(
            target_r=target,
            p_link=p_link,
            pseudo_skills_map=m1_researcher_skills,
            capacity_limit=capacity_limit,
            gain_threshold=gain_threshold,
            stochastic=stochastic,
            cost_multiplier=cost_multiplier
        )

        req = debug["req_skills"]
        covered = set().union(*[m1_researcher_skills[r].intersection(req) for r in team])
        coverage_pct = 100 * len(covered) / max(1, len(req))

        # Ultra goodness
        goodness_ultra = M1.apply_ultra_metric(req, team, m1_researcher_skills)


        results.append({
                    "nsf_proposal_links_v0": p_link,
                    "lead_researcher": target,
                    "team": team,
                    "goodness_score": goodness_ultra
                })
        # results.append({
        #     "proposal_link": p_link,
        #     "target_researcher": target,
        #     "team_size": len(team),
        #     "team": team,
        #     "req_skills": len(req),
        #     "covered_skills": len(covered),
        #     "coverage_pct": coverage_pct,
        #     "goodness_ultra": goodness_ultra,
        #     "cost_constant": debug["cost_constant"],
        # })

        if (idx + 1) % 50 == 0:
            print(f"Processed {idx+1}/{len(proposal_links)} proposals...")

    return pd.DataFrame(results)


# Start small first
subset = all_proposals
results_df = run_teaming_for_proposals(subset, capacity_limit=8, stochastic=True)
display(results_df.head())


[nltk_data] Downloading package wordnet to /Users/tej/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /Users/tej/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


Proposal: https://www.nsf.gov/pubs/2026/nsf26000/nsf26000.htm
Target researcher: Hayley Gutierrez
Team size: 8
Team: ['Hayley Gutierrez', 'William Hogan', 'Michael Barry', 'Krystal Anderson', 'Jesse Bradley', 'Ruben Martinez', 'Michael Brady', 'Megan Daniel DDS']
Cost constant: 0.1603
Req skills: 4
Covered skills: 4
Coverage %: 100.0
Goodness (M1.apply_ultra_metric): 0.7
Processed 50/500 proposals...
Processed 100/500 proposals...
Processed 150/500 proposals...
Processed 200/500 proposals...
Processed 250/500 proposals...
Processed 300/500 proposals...
Processed 350/500 proposals...
Processed 400/500 proposals...
Processed 450/500 proposals...
Processed 500/500 proposals...


,nsf_proposal_links_v0,lead_researcher,team,goodness_score
0,https://www.nsf.gov/pubs/2026/nsf26000/nsf2600...,Hayley Gutierrez,"[Hayley Gutierrez, Megan Daniel DDS, Mark Lewi...",0.7
1,https://www.nsf.gov/pubs/2026/nsf26001/nsf2600...,Robert Smith,"[Robert Smith, Isabel Davis, David Thomas, Ant...",0.7
2,https://www.nsf.gov/pubs/2026/nsf26002/nsf2600...,Marissa Flores,"[Marissa Flores, Donna Stone, Michael Curry, K...",0.7
3,https://www.nsf.gov/pubs/2026/nsf26003/nsf2600...,Brandon Campos,"[Brandon Campos, Leslie Navarro, James Barker,...",0.7
4,https://www.nsf.gov/pubs/2026/nsf26004/nsf2600...,Elizabeth Keller,"[Elizabeth Keller, Brandy Holt, Sabrina Rivera...",0.7


In [8]:
# ==========================================
# === 9) SAVE RESULTS ===
# ==========================================
import os
import datetime

OUT_DIR = "../data/output_data"
os.makedirs(OUT_DIR, exist_ok=True)

# timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

out_csv = f"{OUT_DIR}/Mariginal_utility.csv"
results_df.to_csv(out_csv, index=False)

print("Saved results to:")
print(out_csv)

# Optional sanity check
print("\nSaved columns:")
print(results_df.columns.tolist())
print("Number of rows:", len(results_df))


Saved results to:
../data/output_data/Mariginal_utility.csv

Saved columns:
['nsf_proposal_links_v0', 'lead_researcher', 'team', 'goodness_score']
Number of rows: 500
